# FS-004 / FS-013: Remote GPU Inference — Qwen2.5-VL-3B on Colab

This notebook runs real Qwen2.5-VL-3B-Instruct inference on a free Google Colab T4 GPU
for the MedicalVQA project (FS-004: Medical VQA Model Integration).

## IMPORTANT: Real SLAKE Data Required
- This notebook loads **real SLAKE dataset images and questions** — not synthetic images.
- Synthetic images are **never** used for research baseline predictions.
- Before running, you must upload the SLAKE dataset to Colab (see Step 3).

## Prerequisites
- Google Colab with T4 GPU runtime (Runtime → Change runtime type → T4 GPU)
- Hugging Face account with access to Qwen/Qwen2.5-VL-3B-Instruct (accept license on HF Hub)
- SLAKE dataset downloaded locally, then uploaded to Colab

## No paid APIs or services
All inference runs locally on the Colab T4 GPU using Hugging Face Transformers + bitsandbytes 4-bit quantization.

In [ ]:
# Step 1: Check GPU and environment
import torch
import os
import sys
import shutil

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
else:
    raise RuntimeError("No GPU available! Change runtime to T4 GPU.")

# Check available disk space
total, used, free = shutil.disk_usage("/content")
print(f"Disk total: {total / 1024**3:.1f} GB")
print(f"Disk used: {used / 1024**3:.1f} GB")
print(f"Disk free: {free / 1024**3:.1f} GB")
if free < 30 * 1024**3:
    print("WARNING: Less than 30 GB free disk space. Model download may fail.")

In [ ]:
# Step 2: Install required packages
!pip install -q transformers accelerate bitsandbytes safetensors huggingface_hub pillow numpy

In [ ]:
# Step 3: Set up project code and SLAKE dataset
# Two options:
#   A) Clone from git remote (if repository is accessible)
#   B) Upload project archive/zip to Colab

PROJECT_ROOT = "/content/MedicalVQA"
SLAKE_DIR = "/content/slake"  # User must upload SLAKE dataset here

# Option A: Try cloning from git remote
import subprocess
try:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        capture_output=True, text=True, cwd="/content"
    )
    if result.returncode == 0:
        print(f"Git repo already present at {result.stdout.strip()}")
    else:
        print("No git repo found. You will need to upload project files.")
except FileNotFoundError:
    print("Git not available. Upload project files manually.")

# Check if project root exists
if os.path.exists(PROJECT_ROOT):
    print(f"Project root exists at {PROJECT_ROOT}")
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
        os.chdir(PROJECT_ROOT)
else:
    print(f"WARNING: Project root not found at {PROJECT_ROOT}")
    print("Upload project files to Colab before proceeding.")

# Check if SLAKE directory exists
if os.path.exists(SLAKE_DIR):
    print(f"SLAKE directory exists at {SLAKE_DIR}")
else:
    print(f"WARNING: SLAKE directory not found at {SLAKE_DIR}")
    print("Upload the SLAKE dataset (imgs/ + test.json) to Colab.")

print(f"\nPython path includes project: {PROJECT_ROOT in sys.path}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Step 4: Verify imports (before downloading model weights)
from vlm import QwenVLMClient, MockVLMClient
from vlm.base import VLMClient
from api.remote_worker import RemoteModelWorker
from datasets.slake_loader import SlakeDatasetLoader

print("All imports successful")
print(f"QwenVLMClient: {QwenVLMClient}")
print(f"RemoteModelWorker: {RemoteModelWorker}")
print(f"SlakeDatasetLoader: {SlakeDatasetLoader}")

In [ ]:
# Step 5: Hugging Face authentication (secure, no hardcoded tokens)
from huggingface_hub import login

# Option A: Use Colab secrets (recommended)
#   1. Click the key icon in the left sidebar
   #   2. Add secret: HF_TOKEN = your_huggingface_token
# Option B: Enter token interactively

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HF_AUTH_TOKEN")
if hf_token:
    print("HF_TOKEN found in environment")
    login(token=hf_token, add_to_git_credential=False)
else:
    print("No HF_TOKEN in environment. You may need to run:")
    print("  from huggingface_hub import login")
    print("  login(token='YOUR_TOKEN')")
    print("Also accept the Qwen2.5-VL-3B-Instruct license at:")
    print("  https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct")

In [ ]:
# Step 6: Verify model repository and revision metadata
from huggingface_hub import model_info

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

try:
    info = model_info(MODEL_ID)
    print(f"Model ID: {info.id}")
    print(f"Model sha: {info.sha}")
    print(f"Last modified: {info.lastModified}")
    
    # Estimate download size from siblings
    total_size = sum(s.size for s in info.siblings if s.size)
    print(f"Estimated download size: {total_size / 1024**3:.2f} GB")
    
    if total_size > 40 * 1024**3:
        print("WARNING: Model download exceeds 40 GB. May not fit on Colab disk.")
except Exception as e:
    print(f"Could not fetch model info: {type(e).__name__}: {e}")
    print("Ensure you have accepted the model license on Hugging Face Hub.")

In [ ]:
# Step 7: Load Qwen2.5-VL-3B in 4-bit on T4 GPU
client = QwenVLMClient(
    model_id=MODEL_ID,
    device="cuda",
    quantization="4bit",
    trust_remote_code=True,
)

import time
t0 = time.time()
client._load()
t1 = time.time()

print(f"Model loaded in {t1 - t0:.1f}s")
print(f"Device: {client.device}")

# Report GPU memory
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU allocated: {allocated:.2f} GB")
    print(f"GPU reserved: {reserved:.2f} GB")
    print(f"GPU total: {total:.2f} GB")
    print(f"GPU free: {total - reserved:.2f} GB")
    if total < 8:
        print("WARNING: Less than 8 GB GPU VRAM. 4-bit Qwen2.5-VL-3B may OOM.")

In [ ]:
# Step 8: Load real SLAKE test split
# Uses existing SlakeDatasetLoader — preserves official split, image IDs, question IDs,
# and ground-truth answers. No synthetic images.

IMAGES_DIR = os.path.join(SLAKE_DIR, "imgs")

if not os.path.exists(SLAKE_DIR):
    raise FileNotFoundError(
        f"SLAKE directory not found at {SLAKE_DIR}. "
        "Upload the SLAKE dataset to Colab before running this cell."
    )

loader = SlakeDatasetLoader(
    dataset_dir=SLAKE_DIR,
    images_dir=IMAGES_DIR,
    language="en",
)
samples = loader.load_split("test")

print(f"Loaded {len(samples)} SLAKE test samples (English only)")
for s in samples[:5]:
    print(f"  {s.sample_id}: image={s.image_id}, question='{s.question[:60]}...', gt='{s.answer}'")

In [ ]:
# Step 9: Run real SLAKE baseline inference (10-20 samples)
from PIL import Image
import json

MAX_SAMPLES = 20
results = []

for i, sample in enumerate(samples[:MAX_SAMPLES]):
    print(f"\n--- Sample {i+1}/{min(MAX_SAMPLES, len(samples))}: {sample.sample_id} ---")
    print(f"Image ID: {sample.image_id}")
    print(f"Question: {sample.question}")
    print(f"Ground truth: {sample.answer}")

    # Load real image (not synthetic)
    image_path = sample.image_path
    if not os.path.exists(image_path):
        print(f"SKIP: Image not found at {image_path}")
        results.append({
            "sample_id": sample.sample_id,
            "image_id": sample.image_id,
            "question": sample.question,
            "ground_truth_answer": sample.answer,
            "predicted_answer": "",
            "model": "qwen2.5-vl-3b",
            "device": "cuda",
            "inference_time_s": None,
            "status": "skipped",
            "error": f"Image not found: {image_path}",
        })
        continue

    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as exc:
        print(f"ERROR loading image: {exc}")
        results.append({
            "sample_id": sample.sample_id,
            "image_id": sample.image_id,
            "question": sample.question,
            "ground_truth_answer": sample.answer,
            "predicted_answer": "",
            "model": "qwen2.5-vl-3b",
            "device": "cuda",
            "inference_time_s": None,
            "status": "error",
            "error": f"Failed to load image: {exc}",
        })
        continue

    # Run inference
    t0 = time.time()
    try:
        structured = client.generate_structured(
            marked_image=image,
            question=sample.question,
            region_ids=None,
            temperature=0.0,
            max_tokens=512,
        )
        t1 = time.time()
        inference_time = t1 - t0

        result = {
            "sample_id": sample.sample_id,
            "image_id": sample.image_id,
            "question": sample.question,
            "ground_truth_answer": sample.answer,
            "predicted_answer": structured.answer,
            "model": "qwen2.5-vl-3b",
            "device": "cuda",
            "inference_time_s": round(inference_time, 4),
            "status": "success",
            "findings": [
                {
                    "claim": f.claim,
                    "region_id": f.region_id,
                    "status": f.status,
                    "confidence": f.confidence,
                }
                for f in structured.findings
            ],
        }

        print(f"Predicted answer: {structured.answer[:100]}")
        print(f"Inference time: {inference_time:.2f}s")
        print(f"Findings: {len(structured.findings)}")

    except Exception as exc:
        print(f"ERROR: {type(e).__name__}: {exc}")
        result = {
            "sample_id": sample.sample_id,
            "image_id": sample.image_id,
            "question": sample.question,
            "ground_truth_answer": sample.answer,
            "predicted_answer": "",
            "model": "qwen2.5-vl-3b",
            "device": "cuda",
            "inference_time_s": None,
            "status": "error",
            "error": str(exc),
        }

    results.append(result)

success_count = sum(1 for r in results if r["status"] == "success")
print(f"\n=== Completed {len(results)} samples: {success_count} successful ===")

In [ ]:
# Step 10: Save real SLAKE predictions to JSONL and CSV
import csv
from datetime import UTC, datetime

output_dir = "/content/medicalvqa_results"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
jsonl_path = os.path.join(output_dir, f"qwen_slake_inference_{timestamp}.jsonl")
csv_path = os.path.join(output_dir, f"qwen_slake_inference_{timestamp}.csv")

with open(jsonl_path, "w", encoding="utf-8") as jf:
    for r in results:
        jf.write(json.dumps(r) + "\n")

with open(csv_path, "w", encoding="utf-8", newline="") as cf:
    writer = csv.writer(cf)
    writer.writerow([
        "sample_id", "image_id", "question", "ground_truth_answer",
        "predicted_answer", "model", "device", "inference_time_s",
        "status", "error"
    ])
    for r in results:
        writer.writerow([
            r["sample_id"], r["image_id"], r["question"],
            r["ground_truth_answer"], r["predicted_answer"],
            r["model"], r["device"], r["inference_time_s"],
            r["status"], r.get("error", "")
        ])

print(f"JSONL saved to: {jsonl_path}")
print(f"CSV saved to: {csv_path}")

# Display results
for r in results:
    print(f"\n{r['sample_id']}: {r['status']}")
    print(f"  Q: {r['question']}")
    print(f"  GT: {r['ground_truth_answer']}")
    print(f"  Pred: {r['predicted_answer'][:100]}")

## Optional: Synthetic Smoke Test (SEPARATED from research predictions)

The section below uses synthetic images for a quick smoke test only.
Its outputs are stored separately and MUST NOT be mixed with real SLAKE predictions.

Run this section ONLY to verify the client works before processing real data.

In [ ]:
# Optional smoke test with synthetic images (NOT for research baseline)
from PIL import Image
import numpy as np

smoke_samples = [
    {
        "image_id": "smoke_001",
        "question": "Where is the consolidation located?",
        "ground_truth": "mid-upper zone",
        "image": Image.new("RGB", (512, 512), color=(200, 200, 200)),
    },
]

for sample in smoke_samples:
    arr = np.array(sample["image"])
    arr[150:350, 200:400] = 255
    sample["image"] = Image.fromarray(arr, mode="RGB")

smoke_results = []
for i, sample in enumerate(smoke_samples):
    try:
        structured = client.generate_structured(
            marked_image=sample["image"],
            question=sample["question"],
            region_ids=None,
            temperature=0.0,
            max_tokens=512,
        )
        smoke_results.append({
            "sample_id": f"smoke_{i}",
            "image_id": sample["image_id"],
            "question": sample["question"],
            "ground_truth_answer": sample["ground_truth"],
            "predicted_answer": structured.answer,
            "model": "qwen2.5-vl-3b",
            "device": "cuda",
            "status": "smoke_test_success",
        })
        print(f"Smoke test {i}: {structured.answer[:80]}")
    except Exception as e:
        print(f"Smoke test {i} failed: {e}")

smoke_dir = "/content/medicalvqa_results/smoke_test"
os.makedirs(smoke_dir, exist_ok=True)
smoke_path = os.path.join(smoke_dir, f"smoke_{timestamp}.jsonl")
with open(smoke_path, "w", encoding="utf-8") as f:
    for r in smoke_results:
        f.write(json.dumps(r) + "\n")
print(f"Smoke test results saved to: {smoke_path}")

In [ ]:
# Step 11: GPU memory report (after inference)
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {total:.2f} GB")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved: {reserved:.2f} GB")
    print(f"Peak allocated: {peak:.2f} GB")
    print(f"Free: {total - reserved:.2f} GB")

## Next Steps
- Download the JSONL and CSV results from `/content/medicalvqa_results/`
- Copy them to `D:\Abi\New folder\MedicalVQA\experiments\runs\)
- Do NOT mix smoke test results with real SLAKE predictions
- Run evaluation against ground truth answers
- Proceed to region grounding and uncertainty estimation (later stages)